# Fine-Tune Qwen2.5-7B to Code Like a Pro

**Base Model**: Qwen2.5-7B — strong reasoning and logic, minimal code training  
**Method**: LoRA adapters on full bf16 base  
**Dataset**: 122K Python code instructions (Alpaca format)  
**GPU**: NVIDIA RTX PRO 6000 Blackwell (96GB VRAM), Flash Attention 2, bf16  

In [1]:
!pip install -q \
    transformers \
    peft \
    trl \
    bitsandbytes \
    accelerate \
    datasets

# flash-attn needs --no-cache-dir on this environment (cross-device link bug in pip cache)
!pip install -q flash-attn --no-build-isolation --no-cache-dir

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"bf16 support    : {torch.cuda.is_bf16_supported()}")

PyTorch version : 2.8.0+cu128
CUDA available  : True
GPU             : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM            : 95.0 GB
bf16 support    : True


In [3]:
# ============================================================
# CONFIGURATION — edit these values to experiment
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-7B"
DATASET_NAME = "TokenBender/code_instructions_122k_alpaca_style"

LORA_R = 128
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "all-linear"

NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2      # effective batch = 32
LEARNING_RATE = 2e-4
LR_SCHEDULER = "cosine"
WARMUP_RATIO = 0.03
MAX_SEQ_LENGTH = 2048
OPTIMIZER = "adamw_torch"

LOGGING_STEPS = 10
SAVE_STEPS = 500
OUTPUT_DIR = "./results"
ADAPTER_DIR = f"{OUTPUT_DIR}/lora-adapter"
MERGED_DIR = f"{OUTPUT_DIR}/merged-model"

In [4]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    attn_implementation="flash_attention_2",
    dtype=torch.bfloat16,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded: {MODEL_NAME} (full bf16, no quantization)")
print(f"Memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3:.1f} GB")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-7B (full bf16, no quantization)
Memory footprint: 14.19 GB
VRAM free: 80.8 GB


In [5]:
dataset = load_dataset(DATASET_NAME, split="train")

print(f"Dataset size: {len(dataset):,} examples")
print(f"Columns: {dataset.column_names}\n")

for i in range(3):
    row = dataset[i]
    print(f"--- Example {i+1} ---")
    print(f"Instruction: {row.get('instruction', '')[:200]}")
    print(f"Input:       {row.get('input', '')[:200]}")
    print(f"Output:      {row.get('output', '')[:200]}")
    print()

Dataset size: 121,959 examples
Columns: ['input', 'text', 'instruction', 'output']

--- Example 1 ---
Instruction: Create a function to calculate the sum of a sequence of integers.
Input:       [1, 2, 3, 4, 5]
Output:      # Python code
def sum_sequence(sequence):
  sum = 0
  for num in sequence:
    sum += num
  return sum

--- Example 2 ---
Instruction: Develop a function that will add two strings
Input:       str1 = "Hello "
str2 = "world"
Output:      def add_strings(str1, str2):
    """This function takes two strings and returns the sum of them."""
    return str1 + str2

str1 = "Hello "
str2 = "world"
sum_of_strings = add_strings(str1, str2)

--- Example 3 ---
Instruction: Design a data structure in C++ to store information of a grocery store inventory.
Input:       
Output:      #include <map>
#include <string>

class GroceryStoreInventory {
private:
    std::map<std::string, int> items;
public:
    void addItems(std::string item, int quantity) {
        items[item] = quantit



In [6]:
def format_example(row):
    """Convert Alpaca-style row into Qwen chat template format."""
    instruction = row.get("instruction", "").strip()
    inp = row.get("input", "").strip()
    output = row.get("output", "").strip()

    if inp:
        user_content = f"{instruction}\n\n{inp}"
    else:
        user_content = instruction

    messages = [
        {"role": "system", "content": "You are a highly skilled Python programmer. Write clean, efficient, and correct code."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train set : {len(train_dataset):,} examples")
print(f"Eval  set : {len(eval_dataset):,} examples")
print("\nFormatted example:\n")
print(train_dataset[0]["text"][:1000])

Train set : 115,861 examples
Eval  set : 6,098 examples

Formatted example:

<|im_start|>system
You are a highly skilled Python programmer. Write clean, efficient, and correct code.<|im_end|>
<|im_start|>user
Use NumPy to generate an array of 1000 random numbers between 0 and 9 inclusive.

Not applicable<|im_end|>
<|im_start|>assistant
import numpy as np

arr = np.random.randint(10, size=(1000,)) # array of 1000 random numbers between 0 and 9
print(arr)<|im_end|>



In [7]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 322,961,408 || all params: 7,938,577,920 || trainable%: 4.0683


In [ ]:
EVAL_STEPS = 100

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    optim=OPTIMIZER,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_SEQ_LENGTH,
    packing=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    report_to="none",
    dataset_text_field="text",
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print(f"Total training steps: {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else 'will be computed at train start'}")
print("Starting training...")

trainer.train()

# training results 
'''<div>
      
      <progress value='933' max='933' style='width:300px; height:20px; vertical-align: middle;'></progress>
      [933/933 4:25:47, Epoch 3/3]
    </div>
    <table border="1" class="dataframe">
  <thead>
 <tr style="text-align: left;">
      <th>Step</th>
      <th>Training Loss</th>
      <th>Validation Loss</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>100</td>
      <td>0.758437</td>
      <td>0.740992</td>
    </tr>
    <tr>
      <td>200</td>
      <td>0.748531</td>
      <td>0.736561</td>
    </tr>
    <tr>
      <td>300</td>
      <td>0.739438</td>
      <td>0.734273</td>
    </tr>
    <tr>
      <td>400</td>
      <td>0.730905</td>
      <td>0.734140</td>
    </tr>
    <tr>
      <td>500</td>
      <td>0.720487</td>
      <td>0.733248</td>
    </tr>
    <tr>
      <td>600</td>
      <td>0.717657</td>
      <td>0.732395</td>
    </tr>
    <tr>
      <td>700</td>
      <td>0.724726</td>
      <td>0.734040</td>
    </tr>
    <tr>
      <td>800</td>
      <td>0.710127</td>
      <td>0.734152</td>
    </tr>
    <tr>
      <td>900</td>
      <td>0.739244</td>
      <td>0.733806</td>
    </tr>
    <tr>
      <td>933</td>
      <td>0.721119</td>
      <td>0.733859</td>
    </tr>
  </tbody>
</table><p>'''

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/115861 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/115861 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/115861 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/6098 [00:00<?, ? examples/s]

...

Packing eval dataset:   0%|          | 0/6098 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Total training steps: 0
Starting training...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [10]:
import os

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"\nLoRA adapter saved to: {ADAPTER_DIR}")
print("Saved files:")
for f in sorted(os.listdir(ADAPTER_DIR)):
    size_mb = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1024**2
    print(f"  {f:40s} {size_mb:8.2f} MB")


LoRA adapter saved to: ./results/lora-adapter
Saved files:
  README.md                                    0.00 MB
  adapter_config.json                          0.00 MB
  adapter_model.safetensors                 1232.05 MB
  chat_template.jinja                          0.00 MB
  tokenizer.json                              10.89 MB
  tokenizer_config.json                        0.00 MB
  training_args.bin                            0.01 MB


In [11]:
from transformers import GenerationConfig

del model
del trainer
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
)

merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
eos_id = tokenizer.eos_token_id
gen_config = GenerationConfig(
    eos_token_id=[eos_id, im_end_id],
    pad_token_id=eos_id,
)
gen_config.save_pretrained(MERGED_DIR)

merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"Merged model saved to: {MERGED_DIR}")
print(f"Generation config stop tokens: eos={eos_id}, im_end={im_end_id}")

del base_model
del merged_model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: ./results/merged-model
Generation config stop tokens: eos=151643, im_end=151645


In [12]:
import re

test_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2",
)
test_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

im_end_id = test_tokenizer.convert_tokens_to_ids("<|im_end|>")
stop_token_ids = [test_tokenizer.eos_token_id, im_end_id]

def clean_response(text):
    """Strip trailing non-ASCII garbage and stray tags the model sometimes emits."""
    text = re.sub(r'</?\w+_?\w*>\s*$', '', text)
    text = re.sub(r'[^\x00-\x7F]+\s*$', '', text)
    return text.rstrip()

test_prompts = [
    "Write a Python function that checks if a given string is a palindrome.",
    "Implement a binary search algorithm in Python that returns the index of the target element.",
    "Write a Python class for a stack data structure with push, pop, peek, and is_empty methods.",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are a highly skilled Python programmer. Write clean, efficient, and correct code."},
        {"role": "user", "content": prompt},
    ]
    text = test_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = test_tokenizer(text, return_tensors="pt").to(test_model.device)

    with torch.no_grad():
        output_ids = test_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=stop_token_ids,
        )

    raw = test_tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    response = clean_response(raw)
    print(f"{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    print(response)
    print("\n")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


PROMPT: Write a Python function that checks if a given string is a palindrome.
def is_palindrome(s):
    return s == s[::-1]

print(is_palindrome("racecar"))




Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


PROMPT: Implement a binary search algorithm in Python that returns the index of the target element.
def binary_search(arr, target):
    low = 0
    high = len(arr) - 1
    
    while low <= high:
        mid = (low + high) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
            
    return -1

# Example usage
arr = [1, 3, 5, 7, 9]
target = 5
index = binary_search(arr, target)
print(index)  # Output: 2


PROMPT: Write a Python class for a stack data structure with push, pop, peek, and is_empty methods.
class Stack:
    def __init__(self):
        self.items = []

    def push(self, item):
        self.items.append(item)

    def pop(self):
        if not self.is_empty():
            return self.items.pop()

    def peek(self):
        if not self.is_empty():
            return self.items[-1]

    def is_empty(self):
        return len(self.items) == 0


